# Week 2: Build the churn service

We turn last week's analysis into code that others can call: a small package, a database schema, and an API. Each file is written from a cell, so reading this notebook is reading the code.


## The database

Scores go in a table, so the API doesn't need to recompute them.


In [ ]:
%%writefile schema.sql
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    email VARCHAR(255)
);

CREATE TABLE churn_scores (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER REFERENCES customers(id),
    score NUMERIC(4, 3),
    scored_at TIMESTAMP
);


## Scoring

A customer's risk grows with how long it's been since they last ordered.


In [ ]:
%%writefile churn/__init__.py
"""Scores how likely a customer is to stop buying."""


In [ ]:
%%writefile churn/scoring.py
"""Turns order history into a churn score."""

import pandas as pd


def load_orders(path: str = "../week-1/orders_clean.parquet") -> pd.DataFrame:
    return pd.read_parquet("../week-1/orders_clean.parquet")


def risk_band(score: float) -> str:
    if score > 0.8:
        return "high"
    if score > 0.5:
        return "medium"
    return "low"


def score_customer(days_since_order: int) -> float:
    return min(days_since_order / 365, 1.0)


## The API

One route returns a customer's score and risk band.


In [ ]:
%%writefile churn/api.py
"""The churn API."""

from fastapi import FastAPI
from sqlalchemy import text

from churn.scoring import risk_band, score_customer

app = FastAPI()


@app.get("/churn/{customer_id}")
def churn(customer_id: int, days_since_order: int = 0):
    score = score_customer(days_since_order)
    return {"customer": customer_id, "score": score, "band": risk_band(score)}


@app.get("/health")
def health():
    return {"ok": True}


In [ ]:
%%writefile requirements.txt
fastapi==0.115.0
pandas==2.2.3
sqlalchemy==2.0.36
